# Traffic YOLO — training

7 classes: `Car`, `Number Plate`, `Blur Number Plate`, `Two Wheeler`, `Auto`, `Bus`, `Truck`.
Produces **`best.onnx`** for `web/models/` in the front-end, plus `best.pt`.

Set *Runtime → Change runtime type → **T4 GPU*** before running anything.

In [ ]:
%pip install -q ultralytics onnx onnxslim onnxruntime
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Dataset

Upload `archive.zip`, or drag it into the *Files* panel and skip this cell.
The archive already ships `images/{train,val}` and `labels/{train,val}`; that split is kept.

In [ ]:
import zipfile, yaml
from pathlib import Path

if not Path('archive.zip').exists():
    from google.colab import files
    files.upload()

ROOT = Path('/content/data')
if not ROOT.exists():
    with zipfile.ZipFile('archive.zip') as z:
        z.extractall(ROOT)

BASE = next(p.parent.parent for p in ROOT.rglob('images/train') if p.is_dir())
CLASS_NAMES = ['Car', 'Number Plate', 'Blur Number Plate', 'Two Wheeler', 'Auto', 'Bus', 'Truck']

DATA_YAML = Path('/content/data.yaml')
DATA_YAML.write_text(yaml.safe_dump({
    'path': str(BASE), 'train': 'images/train', 'val': 'images/val',
    'nc': len(CLASS_NAMES), 'names': dict(enumerate(CLASS_NAMES)),
}, sort_keys=False))

for split in ('train', 'val'):
    print(split, len(list((BASE / 'labels' / split).glob('*.txt'))), 'labelled images')

## 2. Training

`yolo11s` is ~20 MB as ONNX, a fair trade-off for in-browser inference.
Use `yolo11n` to target mobile, `yolo11m` for accuracy. Roughly 35 min on a T4.

If `Number Plate` plateaus, raise `IMGSZ` to 960 rather than `EPOCHS`:
small objects are only seen by the finest detection map.

In [ ]:
from ultralytics import YOLO

IMGSZ, EPOCHS = 640, 100

results = YOLO('yolo11s.pt').train(
    data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=16,
    device=0, workers=2, patience=25, seed=42,
    project='/content/runs', name='traffic-yolo', exist_ok=True, plots=True,
    # tuned for road scenes: no vertical flip, mild rotation
    fliplr=0.5, flipud=0.0, degrees=5.0, translate=0.1, scale=0.5, shear=2.0,
    mosaic=1.0, close_mosaic=10,
)
BEST = Path(results.save_dir) / 'weights' / 'best.pt'

## 3. Results

In [ ]:
from IPython.display import Image, display

for name in ('results.png', 'confusion_matrix_normalized.png', 'val_batch0_pred.jpg'):
    p = Path(results.save_dir) / name
    if p.exists():
        print(name); display(Image(filename=str(p), width=900))

m = YOLO(BEST).val(data=str(DATA_YAML), device=0, split='val')

# class_result(i) indexes the evaluated classes, not the absolute class id
order = {int(c): i for i, c in enumerate(m.box.ap_class_index)}
print(f"\n{'class':<20}{'P':>8}{'R':>8}{'mAP50':>10}{'mAP50-95':>10}")
for i, name in enumerate(CLASS_NAMES):
    if i in order:
        p, r, ap50, ap = m.box.class_result(order[i])
        print(f'{name:<20}{p:>8.3f}{r:>8.3f}{ap50:>10.3f}{ap:>10.3f}')
print(f'\nmAP50 {m.box.map50:.4f}   mAP50-95 {m.box.map:.4f}')

## 4. ONNX export

`nms=False` and `dynamic=False`: onnxruntime-web only partially covers the NMS
operators, so the front-end decodes the raw output and runs NMS itself.
`opset=12` for WASM compatibility.

Output must be `[1, 11, 8400]` — 4 box coordinates + 7 classes, over
80² + 40² + 20² candidate positions.

In [ ]:
import onnxruntime as ort

onnx_path = YOLO(BEST).export(format='onnx', imgsz=IMGSZ, opset=12,
                              simplify=True, dynamic=False, nms=False, half=False)

s = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
print('input', s.get_inputs()[0].shape, '-> output', s.get_outputs()[0].shape)
print(f'{Path(onnx_path).stat().st_size / 1e6:.1f} MB')

## 5. Download

Put `best.onnx` in `web/models/` of the front-end, and `best.pt` at the repo root
(`evaluate.py` uses it to report per-class mAP).

In [ ]:
from google.colab import files

files.download(str(onnx_path))
files.download(str(BEST))